# Reto de Web Scraping — Capítulos 3 y 4

**Sitio:** [quotes.toscrape.com](https://quotes.toscrape.com) — creado para practicar scraping.

**Objetivo:** sacar las frases, su autor y sus etiquetas de las 3 primeras páginas,
limpiarlas y guardarlas en un CSV.

Completa los `TODO`. La solución está al final del notebook — no la mires antes de intentarlo 🙂

---

### Pista: así se ve el HTML

Abre la página en Chrome, clic derecho sobre una frase → **Inspeccionar**. Verás:

```html
<div class="quote">
    <span class="text">"La frase..."</span>
    <small class="author">Albert Einstein</small>
    <div class="tags">
        <a class="tag" href="/tag/change/">change</a>
        <a class="tag" href="/tag/thinking/">thinking</a>
    </div>
</div>
```

In [ ]:
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://quotes.toscrape.com/"
HEADERS = {"User-Agent": "BSG-Curso-Scraping/1.0 (ejercicio academico)"}

print("Listo ✔")

---
## Reto 1 — Traer el HTML

In [ ]:
# TODO 1.1: haz el GET a URL, con headers=HEADERS y timeout=10
# r = ...

# TODO 1.2: imprime el status_code (debe ser 200)

# TODO 1.3: arregla el encoding →  r.encoding = r.apparent_encoding

# TODO 1.4: crea la sopa →  sopa = BeautifulSoup(r.text, "html.parser")

---
## Reto 2 — Frase, autor y etiquetas

De cada bloque `div.quote` saca:

| Dato | Dónde está |
|---|---|
| frase | `span.text` |
| autor | `small.author` |
| etiquetas | todos los `a.tag` (únelos con `";"`) |

In [ ]:
frases = []

# TODO 2.1: encuentra todos los bloques  →  sopa.select("div.quote")

# TODO 2.2: recorre los bloques y agrega a 'frases' un diccionario por cada uno

# TODO 2.3: imprime cuántas encontraste (deberían ser 10) y muestra las 3 primeras

---
## Reto 3 — Las 3 primeras páginas

El botón siguiente está en `li.next a` y su `href` es relativo (`/page/2/`).

Recuerda los tres frenos: **tope**, **pausa** y **timeout**.

In [ ]:
MAX_PAGINAS = 3
PAUSA = 1.0

# TODO 3.1: convierte lo del reto 2 en un bucle while con tope MAX_PAGINAS
#           (pista: url = urljoin(url, siguiente["href"]) if siguiente else None)

# TODO 3.2: no olvides el time.sleep(PAUSA) entre páginas

# TODO 3.3: imprime el total acumulado (deberían ser 30 frases)

---
## Reto 4 — Limpiar y guardar

In [ ]:
# TODO 4.1: quita las comillas tipográficas “ ” del texto de cada frase
# TODO 4.2: agrega una columna 'largo' con el número de caracteres de la frase
# TODO 4.3: arma el DataFrame y guarda en frases.csv con encoding="utf-8-sig"
# TODO 4.4: descárgalo a tu PC:
#           from google.colab import files
#           files.download("frases.csv")

---
## Extra (si terminaste antes)

- ¿Qué autor tiene más frases? → `df["autor"].value_counts()`
- ¿Cuáles son las 5 etiquetas más repetidas? → `df["tags"].str.split(";").explode().value_counts()`
- Entra al enlace **(about)** de un autor y trae su fecha de nacimiento.

In [ ]:
# Tu código del extra aquí

---
---
# ⚠️ SOLUCIÓN — no la ejecutes hasta haber intentado el reto

In [ ]:
def extraer_frases(sopa):
    """Devuelve la lista de frases de UNA página ya parseada."""
    resultado = []
    for bloque in sopa.select("div.quote"):
        texto = bloque.select_one("span.text").get_text(strip=True)
        limpio = texto.strip('“”"')                                   # RETO 4.1
        resultado.append({
            "frase": limpio,
            "autor": bloque.select_one("small.author").get_text(strip=True),
            "tags": ";".join(a.get_text(strip=True) for a in bloque.select("a.tag")),
            "largo": len(limpio),                                     # RETO 4.2
        })
    return resultado


url, pagina, frases = URL, 1, []

while url and pagina <= MAX_PAGINAS:
    r = requests.get(url, headers=HEADERS, timeout=10)                # RETO 1.1
    print(f"Página {pagina}: {url} → {r.status_code}")                # RETO 1.2
    r.raise_for_status()
    r.encoding = r.apparent_encoding                                  # RETO 1.3

    sopa = BeautifulSoup(r.text, "html.parser")                       # RETO 1.4
    nuevas = extraer_frases(sopa)                                     # RETO 2
    frases.extend(nuevas)
    print(f"   {len(nuevas)} frases (acumulado: {len(frases)})")

    siguiente = sopa.select_one("li.next a")                          # RETO 3.1
    url = urljoin(url, siguiente["href"]) if siguiente else None
    pagina += 1
    if url:
        time.sleep(PAUSA)                                             # RETO 3.2

print("\nTOTAL:", len(frases), "frases")                              # RETO 3.3

df = pd.DataFrame(frases)
df.to_csv("frases.csv", index=False, encoding="utf-8-sig")            # RETO 4.3
df.head()

In [ ]:
# EXTRA
print("Autores con más frases:")
print(df["autor"].value_counts().head(5), "\n")

etiquetas = df["tags"].str.split(";").explode()
print("Etiquetas más repetidas:")
print(etiquetas[etiquetas != ""].value_counts().head(5))